# Rung 0 — recomputing every claim from the run's own artifacts

This notebook *proves*; `summary.ipynb` *explains*. Nothing here is imported from this project:
every number below is recomputed from a committed table using the Python standard library
(`csv`, `gzip`, `hashlib`, `statistics`) and arithmetic written out in full, so what you read is
exactly what is computed. A notebook that only called the verification script would relocate the
trust rather than discharge it; that script appears once, in the last cell, as a cross-check.

**What this rung measures.** For each (cell line, drug, dose) triple with two or more plates,
the plate ids are sorted and assigned alternately to two groups, each group's per-gene log2 fold
change is averaged, and the two averaged profiles are correlated across genes. That correlation is
computed twice from the same split: over **all** genes, and over the **responder** genes the
triple's *first* group called differentially expressed. Both are corrected to full length by
Spearman-Brown, `2r / (1 + r)`, and read against mismatched-condition floors. Every statistic
below therefore comes in two families, `all_*` and `responder_*`, in one summary row. Beside them
the run commits a noise decomposition — the between-plate share of the fold-change variance,
pooled over gene-conditions and floored once — and a dose-strata table carrying every candidate
aggregate of the per-triple correlations.

**What cannot be checked here, stated rather than hidden.** The 1,026 Tahoe pseudobulk shards
live on cluster scratch and are far too large to commit, so shard integrity reduces here to the
committed manifest's content hash. The committed per-gene noise table is a two-million-row sample;
no promoted noise number is read from it — they are recomputed from the per-condition sums, which
cover every gene-condition. A promoted copy under `results/` does not exist until after gate 2,
and the permutation check is a separate cluster job; where those are absent the cell says so and
moves on, rather than failing on the calendar.

**Before the run.** The artifacts are uncommitted between the run and promotion (PROCESS, "What
reaches GitHub, and when"). Every cell degrades to "artifact not present yet" until they arrive,
so this notebook executes end to end on a fresh checkout. Set `RUNG0_TASK_DIR` to point it at
another run's artifacts.

In [ ]:
import contextlib
import csv
import gzip
import hashlib
import json
import math
import os
import statistics
import subprocess
import sys
from pathlib import Path


def find_repo(start: Path) -> Path:
    """The repository root: the first ancestor carrying pyproject.toml."""
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    return start


REPO = find_repo(Path.cwd().resolve())
TASK = "rung0-assay-reliability"
TASK_DIR = Path(os.environ.get("RUNG0_TASK_DIR") or (REPO / "docs" / "tasks" / TASK))
TRANCHE = "tahoe100m-pseudobulk-de.v1"

#: gene set -> the per-condition table column holding that set's correlation
GENE_SETS = {"all": "r", "responder": "r_responder"}
KEYS = ("patient", "drug", "dose")

#: Declared in design.md, one per measurement step plus the per-gene diagnostic and the dose
#: figure. The two permutation figures are expected only when that separate job ran.
FIGURES = [
    "01_build.png",
    "02_split.png",
    "03_select.png",
    "04_score.png",
    "05_decompose.png",
    "06_null.png",
    "07_terciles.png",
    "08_power.png",
    "09_per_gene_reliability.png",
    "10_dose.png",
]
PERMUTATION_FIGURES = {
    "11_permutation_vs_bootstrap.png": "rung0_permutation_summary.csv",
    "11_permutation_vs_bootstrap_responder.png": "rung0_permutation_summary_responder.csv",
}

print(f"repository  {REPO}")
print(f"task dir    {TASK_DIR}")
print(f"present     {(TASK_DIR / 'rung0_reliability.csv').exists()}")

### The helpers, written out once

A handful of small functions and nothing else: read a table (plain or gzipped) into a list of
dictionaries, turn a cell into a number, decide whether a reported value is a recomputed one
rounded to its last printed place, and print claim / recomputed / verdict. `read_rows` returns
`None` when the run has not happened yet, which is what lets every cell below degrade to a message
instead of a traceback.

One detail that matters for correctness: one of the screen's fifty cell lines has a missing DepMap
identifier and appears throughout as the literal string `NA`. The `csv` module keeps it as text,
which is why the keys join to themselves here. A second: dose is a number in every table, and a
number written as `5.0` in one and `5` in another is the same dose, so keys are joined on the
number rather than on the text.

In [ ]:
def read_rows(name, directory=None):
    """Rows of a committed table as dictionaries, or None when it has not been written yet."""
    path = (directory or TASK_DIR) / name
    if not path.exists():
        print(f"artifact not present yet: {path}")
        return None
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", newline="") as handle:
        return list(csv.DictReader(handle))


def num(cell):
    """A CSV cell as a float; an empty cell is a missing value, written as nan."""
    text = (cell or "").strip()
    return float("nan") if text == "" else float(text)


def close(claim, recomputed, decimals):
    """True when the reported value is the recomputed one rounded to `decimals` places."""
    if math.isnan(claim) or math.isnan(recomputed):
        return math.isnan(claim) and math.isnan(recomputed)
    return abs(claim - recomputed) <= 0.5 * 10**-decimals + 1e-12


def key_of(row):
    """The (line, drug, dose) key of a row, with dose as a number so 5 and 5.0 agree."""
    dose = row["dose"].strip()
    with contextlib.suppress(ValueError):
        dose = float(dose)
    return (row["patient"], row["drug"], dose)


def flag(cell):
    return (cell or "").strip().lower() in ("true", "1")


def verdict(name, claim, recomputed, ok):
    print(f"{name}\n  claim      : {claim}\n  recomputed : {recomputed}")
    print(f"  verdict    : {'PASS' if ok else 'FAIL'}\n")


def sha256_of(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


summary_rows = read_rows("rung0_reliability.csv")
summary = summary_rows[0] if summary_rows else None
per_pair = read_rows("rung0_per_pair_r.csv")

## Claim 1 — the reported count is the number of triples actually scored

`n_pairs` is the number of triples whose correlation is finite, not the number of rows in the
per-condition table. A triple that fell below the fifty-gene scoring threshold is kept in the
table, honestly blank, and is not one of the triples the mean is taken over. The two families
differ here: a triple can be scoreable over all genes and unscoreable over its responders.

In [ ]:
if summary and per_pair:
    for label, column in GENE_SETS.items():
        values = [num(row[column]) for row in per_pair]
        finite = [v for v in values if not math.isnan(v)]
        verdict(
            f"{label}: n_pairs is the count of finite per-condition correlations",
            f"reported n_pairs {summary[f'{label}_n_pairs']}",
            f"{len(finite)} finite of {len(per_pair)} rows in rung0_per_pair_r.csv",
            len(finite) == int(summary[f"{label}_n_pairs"]),
        )

## Claim 2 — the mean, median and quartiles are the ones the per-condition values give

The declared statistic is the mean over triples of the per-triple correlation, with the median
and quartiles beside it because a mean alone cannot say whether the reproducibility is spread
evenly or carried by a few triples. All four are recomputed here from the same committed values,
using the standard library's quantiles (inclusive method, which is the interpolation the run
used) rather than any library the run itself called.

In [ ]:
if summary and per_pair:
    for label, column in GENE_SETS.items():
        finite = [num(row[column]) for row in per_pair]
        finite = [v for v in finite if not math.isnan(v)]
        q1, median, q3 = statistics.quantiles(finite, n=4, method="inclusive")
        for what, key, value in (
            ("mean", "splithalf_mean_r", statistics.fmean(finite)),
            ("median", "splithalf_median_r", median),
            ("lower quartile", "splithalf_q1_r", q1),
            ("upper quartile", "splithalf_q3_r", q3),
        ):
            verdict(
                f"{label}: {what} recomputes from the per-condition correlations",
                f"reported {key} {summary[f'{label}_{key}']}",
                f"over {len(finite)} committed values: {value:.4f}",
                close(num(summary[f"{label}_{key}"]), value, 3),
            )

## Claim 3 — Spearman-Brown is `2r / (1 + r)` of that mean, and again on the equal-halves subset

A split-half correlation is a reliability at half length. `2r / (1 + r)` corrects it to the full
screen's length, and the correction assumes the two halves are the same size. Under the
alternating split that is exactly an even plate count, which most replicated triples have; the
corrected value is still reported a second time over that subset so the assumption is read off
the data rather than argued about.

The correction is applied to the mean over triples, not per triple and then averaged: `2r /
(1 + r)` is not linear, so those two differ, and the design's declared statistic is the mean.

In [ ]:
if summary and per_pair:
    even_flag = [flag(row["n_plates_even"]) for row in per_pair]
    for label, column in GENE_SETS.items():
        values = [num(row[column]) for row in per_pair]
        finite = [v for v in values if not math.isnan(v)]
        even = [v for v, f in zip(values, even_flag, strict=True) if f and not math.isnan(v)]
        mean = statistics.fmean(finite)
        corrected = 2 * mean / (1 + mean)
        verdict(
            f"{label}: Spearman-Brown correction is 2r/(1+r) of that mean",
            f"reported spearman_brown_full {summary[f'{label}_spearman_brown_full']}",
            f"2 x {mean:.4f} / (1 + {mean:.4f}) = {corrected:.4f}",
            close(num(summary[f"{label}_spearman_brown_full"]), corrected, 3),
        )
        mean_even = statistics.fmean(even) if even else float("nan")
        corrected_even = 2 * mean_even / (1 + mean_even) if even else float("nan")
        verdict(
            f"{label}: the equal-halves subset is counted and corrected the same way",
            f"reported n_pairs_even {summary[f'{label}_n_pairs_even']}, mean "
            f"{summary[f'{label}_splithalf_mean_r_even_plates']}, corrected "
            f"{summary[f'{label}_spearman_brown_full_even_plates']}",
            f"{len(even)} equal-halves triples, mean {mean_even:.4f}, "
            f"corrected {corrected_even:.4f}",
            len(even) == int(summary[f"{label}_n_pairs_even"])
            and close(num(summary[f"{label}_splithalf_mean_r_even_plates"]), mean_even, 3)
            and close(num(summary[f"{label}_spearman_brown_full_even_plates"]), corrected_even, 3),
        )

## Claim 4 — the positive fraction is the share of triples above zero

`frac_pos` says how much of the reproducibility is general rather than carried by a handful of
triples. It is a plain count over the same committed values.

In [ ]:
if summary and per_pair:
    for label, column in GENE_SETS.items():
        finite = [num(row[column]) for row in per_pair]
        finite = [v for v in finite if not math.isnan(v)]
        positive = sum(1 for v in finite if v > 0)
        verdict(
            f"{label}: frac_pos is the share of conditions above zero",
            f"reported frac_pos {summary[f'{label}_frac_pos']}",
            f"{positive} of {len(finite)} above zero: {positive / len(finite):.4f}",
            close(num(summary[f"{label}_frac_pos"]), positive / len(finite), 3),
        )

## Claim 5 — each chance floor is the mean of the mismatched-condition draws behind it

A floor is what a correlation of this kind is worth by construction alone. Three strata: *any
pair*; *different drug and line*, the generic-structure floor; and *same drug at the same dose,
different line*, the stricter floor, since two lines given one drug at one dose share that
drug's generic response. The summary reports only each stratum's mean, so the run exported every
individual draw and the means are recomputed from them here. `null_n_draws` is the size of the
stratum the p-value is read against.

In [ ]:
draws = read_rows("rung0_null_draws.csv")
if summary and draws:
    for label in GENE_SETS:
        subset = [row for row in draws if row["gene_set"] == label]
        counts = {}
        for stratum in ("any_pair", "diff_drug", "same_drug"):
            values = [num(row["r"]) for row in subset if row["stratum"] == stratum]
            counts[stratum] = len(values)
            mean = statistics.fmean(values) if values else float("nan")
            key = f"{label}_null_{stratum}_mean_r"
            verdict(
                f"{label}: the {stratum} floor recomputes from its draws",
                f"reported {key} {summary[key]}",
                f"mean of {len(values)} committed draws: {mean:.4f}",
                bool(values) and close(num(summary[key]), mean, 3),
            )
        expected = counts["diff_drug"] or counts["any_pair"]
        verdict(
            f"{label}: null_n_draws counts the stratum the p-value is read against",
            f"reported null_n_draws {summary[f'{label}_null_n_draws']}",
            f"{counts['diff_drug']} diff_drug draws (any_pair fallback: {counts['any_pair']})",
            expected == int(summary[f"{label}_null_n_draws"]),
        )

## Claim 6 — the observed mean clears both floors, and each comparison states its power

A reliability is only a reliability if it sits above what mismatched conditions give for free, so
the observed mean is required to exceed both floors it is read against. Beside it, each comparison
carries its minimum detectable effect at α = 0.05 and power 0.80 from the same bootstrap as its
p-value: a null result without its MDE cannot be told apart from an underpowered one, which is the
distinction the later rungs' small cohorts turn on. Here the MDEs are required to exist — positive
and finite — and the MDE curve's observed row is required to be the summary's own count and MDE.

In [ ]:
curve = read_rows("rung0_mde_curve.csv")
if summary:
    for label in GENE_SETS:
        mean = num(summary[f"{label}_splithalf_mean_r"])
        floor_diff = num(summary[f"{label}_null_diff_drug_mean_r"])
        floor_same = num(summary[f"{label}_null_same_drug_mean_r"])
        verdict(
            f"{label}: the observed mean clears both floors",
            "mean > different-drug floor and > same-drug floor",
            f"{mean} > {floor_diff} and {mean} > {floor_same}",
            mean > floor_diff and mean > floor_same,
        )
        mdes = [
            num(summary[f"{label}_mde_80_vs_diff_drug"]),
            num(summary[f"{label}_mde_80_vs_same_drug"]),
        ]
        verdict(
            f"{label}: both minimum detectable effects are positive and finite",
            "an MDE at alpha 0.05, power 0.80 against each floor",
            f"vs different-drug {mdes[0]}, vs same-drug {mdes[1]}",
            all(math.isfinite(m) and m > 0 for m in mdes),
        )
        if curve:
            observed = [r for r in curve if r["gene_set"] == label and flag(r["observed"])]
            n = int(observed[0]["n_pairs"]) if observed else -1
            mde = num(observed[0]["mde"]) if observed else float("nan")
            verdict(
                f"{label}: the MDE curve's observed row is the summary's count and MDE",
                f"reported n_pairs {summary[f'{label}_n_pairs']}, "
                f"mde {summary[f'{label}_mde_80_vs_diff_drug']}",
                f"rung0_mde_curve.csv: {len(observed)} observed row(s), n_pairs {n}, mde {mde}",
                len(observed) == 1
                and n == int(summary[f"{label}_n_pairs"])
                and close(num(summary[f"{label}_mde_80_vs_diff_drug"]), mde, 4),
            )

## Claim 7 — reproducibility rises with effect size, under both rankings (the in-run control)

Triples are cut into thirds by the response size ONE half measured, and the correlation of the
pair is averaged within each third; then the halves swap. Ranking by one half alone is what
makes this a control: under no signal the other half is independent of the ranking and every
third sits at zero, whereas ranking by the two halves' sum selects triples whose halves happened
to agree and pure noise rises. Both rankings are recomputed here from the per-condition table's
own half magnitudes, compared with the committed tercile table, and both must rise. A failure to
rise is a finding about the screen rather than a bug in this notebook — which is why the means
are printed whatever the verdict.

In [ ]:
terciles = read_rows("rung0_effect_terciles.csv")
if terciles and per_pair:
    r_all = [num(row["r"]) for row in per_pair]
    for ranked_by in ("half0", "half1"):
        magnitude = [num(row[f"mean_abs_{ranked_by}"]) for row in per_pair]
        pairs = [
            (m, r)
            for m, r in zip(magnitude, r_all, strict=True)
            if math.isfinite(m) and math.isfinite(r)
        ]
        mags = sorted(m for m, _ in pairs)
        # numpy's default quantile is linear interpolation between order statistics, which is
        # the standard library's "inclusive" method
        lo, hi = statistics.quantiles(mags, n=3, method="inclusive")
        thirds = [[], [], []]
        for m, r in pairs:
            thirds[0 if m <= lo else (1 if m <= hi else 2)].append(r)
        recomputed = [statistics.fmean(t) if t else float("nan") for t in thirds]
        reported = [
            num(row["mean_r"])
            for row in sorted(
                (t for t in terciles if t["ranked_by"] == ranked_by),
                key=lambda t: int(t["tercile"]),
            )
        ]
        verdict(
            f"terciles ranked by {ranked_by}: the means recompute from the per-condition table",
            "reported " + " -> ".join(f"{m:.4f}" for m in reported),
            "recomputed " + " -> ".join(f"{m:.4f}" for m in recomputed),
            len(reported) == 3
            and all(close(a, b, 4) for a, b in zip(reported, recomputed, strict=True)),
        )
        verdict(
            f"reproducibility rises with effect size, ranked by {ranked_by}",
            "tercile 1 < tercile 2 < tercile 3 of the split-half mean",
            " -> ".join(f"{m:.4f}" for m in recomputed),
            recomputed[0] < recomputed[1] < recomputed[2],
        )

## Claim 8 — selecting responders from both halves inflates a correlation out of nothing

The responder genes are chosen from the first plate group alone. Choosing them from the two halves
pooled would inflate the correlation by winner's curse: writing the halves as *a* and *b*, their
sum and difference are independent, so selecting on a large `|a + b|` inflates `var(a + b)` alone
and `cov(a, b) = (var(a+b) - var(a-b)) / 4` goes positive with nothing generating it. The run
measures both rules on a pool with no signal at all, and the pooled rule must come back visibly
higher. That gap is the size of the error the one-sided rule avoids.

In [ ]:
leakage = read_rows("rung0_leakage_control.csv")
if leakage:
    by_rule = {row["rule"]: num(row["mean_r"]) for row in leakage}
    one, pooled = by_rule.get("one-sided", float("nan")), by_rule.get("pooled", float("nan"))
    verdict(
        "two-sided selection inflates a signal-free correlation, one-sided does not",
        "pooled mean r > one-sided mean r on a pool with no signal",
        f"pooled {pooled} vs one-sided {one}",
        math.isfinite(one) and math.isfinite(pooled) and pooled > one,
    )

## Claim 9 — the noise decomposition, pooled from the per-condition sums

`lfcSE` is the standard error of one plate's treated-versus-control contrast: cell-sampling error
at that row's cell counts. It cannot see plate-to-plate variation — culture day, handling,
position. Across plates at a fixed dose the fold change varies by both, so

    sigma2_plate = mean over gene-conditions of var_across_plates(log2FoldChange)
                   - mean over gene-conditions of lfcSE^2,  floored at zero once

estimates the plate component alone, and the between-plate share is that over the mean variance.
The floor comes AFTER the average: at two plates each gene's variance has one degree of freedom,
and flooring each gene first would report a share of about 0.15 from nothing.

The run commits every condition's gene count and mean variance and mean squared standard error,
so the pooled share for all genes, for the responders, and the mean over conditions of each
condition's own share all recompute here from sums that cover every gene-condition. The committed
per-gene table is a sample; it serves the row-wise identity `sigma2_plate_signed = var - se^2`,
the strata table, and a control pool that planted a share of one half at two plates.

In [ ]:
noise_summary = read_rows("rung0_noise_decomposition.csv")
by_condition = read_rows("rung0_noise_by_condition.csv")
if noise_summary and by_condition:
    reported = noise_summary[0]

    def pooled(n_col, var_col, se2_col):
        n = [num(r[n_col]) for r in by_condition]
        var = [num(r[var_col]) for r in by_condition]
        se2 = [num(r[se2_col]) for r in by_condition]
        keep = [
            i for i in range(len(n)) if n[i] > 0 and math.isfinite(var[i]) and math.isfinite(se2[i])
        ]
        total = sum(n[i] for i in keep)
        mean_var = sum(n[i] * var[i] for i in keep) / total
        mean_se2 = sum(n[i] * se2[i] for i in keep) / total
        return max(mean_var - mean_se2, 0.0) / mean_var, int(total)

    share, n_all = pooled("n_gene_doses", "var_lfc_mean", "mean_se2_mean")
    share_resp, _ = pooled(
        "n_responder_gene_doses", "var_lfc_mean_responders", "mean_se2_mean_responders"
    )
    per_cond = [num(r["between_plate_fraction_pooled"]) for r in by_condition]
    per_cond = [v for v in per_cond if math.isfinite(v)]
    verdict(
        "noise: n_gene_conditions is the sum of the per-condition gene counts",
        f"reported n_gene_conditions {reported['n_gene_conditions']}",
        f"sum over {len(by_condition)} conditions: {n_all}",
        n_all == int(num(reported["n_gene_conditions"])),
    )
    verdict(
        "noise: the pooled between-plate share recomputes from the per-condition sums",
        f"reported between_plate_fraction_pooled {reported['between_plate_fraction_pooled']}",
        f"from the per-condition means and counts: {share:.6f}",
        close(num(reported["between_plate_fraction_pooled"]), share, 4),
    )
    verdict(
        "noise: the responders' pooled share recomputes from the responders' sums",
        f"reported {reported['between_plate_fraction_pooled_responders']}",
        f"{share_resp:.6f}",
        close(num(reported["between_plate_fraction_pooled_responders"]), share_resp, 4),
    )
    verdict(
        "noise: the over-conditions share is the mean of the per-condition pooled shares",
        f"reported {reported['between_plate_fraction_pooled_over_conditions']}",
        f"mean of {len(per_cond)} per-condition shares: {statistics.fmean(per_cond):.6f}",
        close(
            num(reported["between_plate_fraction_pooled_over_conditions"]),
            statistics.fmean(per_cond),
            4,
        ),
    )

In [ ]:
noise_path = TASK_DIR / "rung0_noise_per_gene.csv.gz"
sample = []
if noise_summary and noise_path.exists():
    reported = noise_summary[0]
    worst = 0.0
    with gzip.open(noise_path, "rt", newline="") as handle:
        for row in csv.DictReader(handle):
            var, se2 = num(row["var_lfc"]), num(row["mean_se2"])
            signed = num(row["sigma2_plate_signed"])
            if math.isfinite(var) and math.isfinite(se2):
                worst = max(worst, abs((var - se2) - signed))
            sample.append((var, se2, num(row["base_mean"]), abs(num(row["mean_lfc"]))))
    n_expected = int(num(reported["n_sample_rows"])) if "n_sample_rows" in reported else len(sample)
    verdict(
        "noise: sigma2_plate_signed = var_lfc - mean_se2 on every committed sample row",
        f"{n_expected} committed sample rows, the identity on every one",
        f"{len(sample)} rows read; worst absolute deviation {worst:.3e}",
        len(sample) == n_expected and worst < 1e-9,
    )
elif noise_summary:
    print(f"artifact not present yet: {noise_path}")

strata = read_rows("rung0_noise_strata.csv")
if strata and sample:
    # The run cut the sample into quartiles of the rank of baseMean and of |mean lfc| (ties by
    # order of appearance) with right-inclusive edges at linear-interpolation quantiles of the
    # ranks: rank r falls in the k-th quartile where k counts the edges 1 + (n-1)j/4 below it.
    rows = [
        (v, s, b, a, i)
        for i, (v, s, b, a) in enumerate(sample)
        if math.isfinite(v) and v > 0 and math.isfinite(s)
    ]
    n = len(rows)

    def quartiles(values):
        order = sorted(range(n), key=lambda i: (values[i], i))
        rank = [0] * n
        for position, i in enumerate(order):
            rank[i] = position + 1
        edges = [1 + (n - 1) * j / 4 for j in (1, 2, 3)]
        return [1 + sum(1 for e in edges if r > e) for r in rank]

    expr_q = quartiles([b for _, _, b, _, _ in rows])
    resp_q = quartiles([a for _, _, _, a, _ in rows])
    cells = {}
    for (v, s, _, _, _), e, q in zip(rows, expr_q, resp_q, strict=True):
        c = cells.setdefault((e, q), [0, 0.0, 0.0])
        c[0] += 1
        c[1] += v
        c[2] += s
    bad = []
    for row in strata:
        key = (int(row["expression_quartile"]), int(row["response_quartile"]))
        c = cells.get(key)
        if c is None or c[0] != int(row["n"]):
            bad.append(f"{key}: count")
            continue
        mean_var, mean_se2 = c[1] / c[0], c[2] / c[0]
        share = max(mean_var - mean_se2, 0.0) / mean_var
        if not close(num(row["between_plate_fraction_pooled"]), share, 4):
            bad.append(f"{key}: share")
    verdict(
        "noise: each stratum's pooled share recomputes from the committed sample",
        f"{len(strata)} strata rows, each a count and a pooled share",
        "every stratum recomputes"
        if not bad and len(strata) == len(cells)
        else f"disagreeing: {bad}",
        not bad and len(strata) == len(cells) > 0,
    )

control = read_rows("rung0_control_noise.csv.gz")
if control:
    var = [num(r["var_lfc"]) for r in control]
    se2 = [num(r["mean_se2"]) for r in control]
    keep = [
        i for i in range(len(var)) if math.isfinite(var[i]) and var[i] > 0 and math.isfinite(se2[i])
    ]
    mean_var = statistics.fmean(var[i] for i in keep)
    share = max(mean_var - statistics.fmean(se2[i] for i in keep), 0.0) / mean_var
    verdict(
        "noise: the control pool's pooled share recovers the planted one half",
        "planted share 0.5 at two plates",
        f"pooled over {len(keep)} control rows: {share:.4f}",
        abs(share - 0.5) < 0.03,
    )

## Claim 10 — every example scatter reproduces the correlation it is shown under

Every correlation in this analysis is one point in a distribution, so the run exports both halves'
per-gene values for a few example triples spanning the reliability range, plus the two
mismatched comparisons the floors are built from. Recomputing each correlation from its own
exported points is what stops a caption asserting a number the plotted data does not support.

In [ ]:
profiles = read_rows("rung0_example_pair_profiles.csv.gz")
index = read_rows("rung0_example_pair_index.csv")
if profiles and index:
    points = {}
    for row in profiles:
        points.setdefault(row["example_id"], []).append((num(row["lfc0"]), num(row["lfc1"])))
    for entry in index:
        pairs = points.get(entry["example_id"], [])
        xs = [x for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        ys = [y for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        recomputed = statistics.correlation(xs, ys) if len(xs) > 1 else float("nan")
        verdict(
            f"example {entry['example_id']} ({entry['kind']}) reproduces its correlation",
            f"index r_shown {entry['r_shown']} over {entry['n_genes_shown']} genes",
            f"from the {len(pairs)} committed points: {recomputed:.4f}",
            close(num(entry["r_shown"]), recomputed, 4)
            and len(pairs) == int(entry["n_genes_shown"]),
        )

## Claim 11 — every declared figure exists, and the score figure's printed number reproduces

`design.md` names the figures before the run, so a reviewer sees the evidence the design promised
rather than the subset that looked best afterwards. Each is required to be a real PNG, not an empty
placeholder; the two permutation figures are expected exactly when that job's summary exists. The
score figure additionally writes the points it drew and the correlation it printed on each panel
— one panel per example and gene set — to a companion table, so the number on the image is
recomputable from the image's own data.

In [ ]:
figure_dir = TASK_DIR / "figures"
if figure_dir.exists():
    expected = list(FIGURES) + [
        f for f, src in PERMUTATION_FIGURES.items() if (TASK_DIR / src).exists()
    ]
    present = [name for name in expected if (figure_dir / name).exists()]
    real = [
        name
        for name in present
        if (figure_dir / name).stat().st_size > 5_000
        and (figure_dir / name).read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"
    ]
    verdict(
        "every figure design.md declares was written, and is a real image",
        f"{len(expected)} figures, each a PNG over 5 kB",
        f"{len(present)} present, {len(real)} non-trivial"
        + (
            f"; missing {sorted(set(expected) - set(present))}"
            if len(present) < len(expected)
            else ""
        ),
        len(real) == len(expected),
    )
else:
    print(f"artifact not present yet: {figure_dir}")

values = read_rows("04_score.values.csv.gz", directory=figure_dir)
if values:
    panels = {}
    printed = {}
    for row in values:
        panel = row["example_id"] + " | " + row.get("gene_set", "")
        panels.setdefault(panel, []).append((num(row["lfc0"]), num(row["lfc1"])))
        printed[panel] = num(row["r_printed"])
    for panel, pairs in panels.items():
        xs = [x for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        ys = [y for x, y in pairs if math.isfinite(x) and math.isfinite(y)]
        recomputed = statistics.correlation(xs, ys) if len(xs) > 1 else float("nan")
        verdict(
            f"score figure panel {panel} prints the correlation of its own points",
            f"printed r {printed[panel]:.4f}",
            f"from the {len(pairs)} plotted points: {recomputed:.4f}",
            close(printed[panel], recomputed, 4),
        )

## Claim 12 — the split, the pool, and the scored triples agree with one another

`rung0_split_assignment.csv` is the split itself: one row per (line, drug, dose, plate) with the
half the plate went to. The rule is checked directly — plates sorted by id as text within a triple
alternate 0, 1, 0, 1 — and the counts it implies must be the pool description's. A triple reaches
the per-condition table exactly when it has two or more plates, so the counts must agree; and the
equal-halves flag, recorded in both tables and defining the Spearman-Brown subset above, must be
the same flag in both places on the full (line, drug, dose) key and must mean exactly an even
plate count.

In [ ]:
pool = read_rows("rung0_pool_description.csv")
split = read_rows("rung0_split_assignment.csv")
if pool and per_pair and split:
    by_triple = {}
    for row in split:
        by_triple.setdefault(key_of(row), []).append((row["plate"], int(row["half"])))
    breaks = 0
    counts = {}
    for key, plates in by_triple.items():
        ordered = sorted(plates, key=lambda p: p[0])  # plate ids are text; sorted as text
        for rank, (_, half) in enumerate(ordered):
            if half != rank % 2:
                breaks += 1
        counts[key] = (
            len(plates),
            sum(1 for _, h in plates if h == 0),
            sum(1 for _, h in plates if h == 1),
        )
    verdict(
        "the split alternates over sorted plate ids within every triple",
        f"{len(split)} (triple, plate) rows: half = rank of the plate within its triple, mod 2",
        f"{breaks} rows break the rule",
        breaks == 0 and len(split) > 0,
    )
    pool_by_key = {key_of(row): row for row in pool}
    count_disagree = sum(
        1
        for key, (n, h0, h1) in counts.items()
        if key not in pool_by_key
        or (
            int(pool_by_key[key]["n_plates"]),
            int(pool_by_key[key]["n_plates_half0"]),
            int(pool_by_key[key]["n_plates_half1"]),
        )
        != (n, h0, h1)
    )
    verdict(
        "the pool's plate counts per half are the split assignment's",
        f"{len(pool)} triples in the pool description",
        f"{len(counts)} triples in the split; {count_disagree} disagree",
        count_disagree == 0 and len(counts) == len(pool),
    )
    replicated = [row for row in pool if int(row["n_plates"]) >= 2]
    pair_keys = {key_of(row) for row in per_pair}
    joined = sum(1 for row in pool if key_of(row) in pair_keys)
    verdict(
        "the scored conditions are the replicated triples, on the full key",
        f"{len(replicated)} triples with two or more plates",
        f"{len(per_pair)} per-condition rows, {joined} joined on (line, drug, dose)",
        len(replicated) == len(per_pair) == joined,
    )
    pool_even = {key_of(row): flag(row["n_plates_even"]) for row in pool}
    agree = sum(1 for row in per_pair if pool_even.get(key_of(row)) == flag(row["n_plates_even"]))
    verdict(
        "the equal-halves flag agrees between the pool and the per-condition table",
        "n_plates_even is one flag, recorded twice, joined on the full key",
        f"{agree} of {len(per_pair)} triples agree",
        agree == len(per_pair),
    )
    equal = all(
        flag(r["n_plates_even"]) == (int(r["n_plates_half0"]) == int(r["n_plates_half1"]))
        for r in pool
    )
    parity = all(flag(r["n_plates_even"]) == (int(r["n_plates"]) % 2 == 0) for r in pool)
    verdict(
        "the equal-halves flag means equal halves, which is an even plate count",
        "n_plates_even == (half0 == half1) == (n_plates is even)",
        f"equal halves: {equal}; even count: {parity}",
        equal and parity,
    )

## Claim 13 — every candidate ceiling in the dose-strata table recomputes from the triples

Holding dose fixed made the unit a triple, and the screen did not replicate its doses evenly, so
what the summary row's mean weights is a declared choice. The strata table carries every
candidate — each dose level alone, all triples equally, each (line, drug) pair once with its
triples averaged first — so the choice is read off committed numbers. Each row is re-derived
here, and the summary row's mean is required to be the all-triples, equal-weight row.

In [ ]:
dose_strata = read_rows("rung0_dose_strata.csv")
if dose_strata and per_pair and summary:
    for label, column in GENE_SETS.items():
        r = [num(row[column]) for row in per_pair]
        finite = [i for i, v in enumerate(r) if math.isfinite(v)]
        bad = []
        pooled_mean = float("nan")
        for row in (s for s in dose_strata if s["gene_set"] == label):
            dose, weighting = row["dose"].strip(), row["weighting"]
            if weighting == "per_line_drug":
                groups = {}
                for i in finite:
                    groups.setdefault((per_pair[i]["patient"], per_pair[i]["drug"]), []).append(
                        r[i]
                    )
                values = [statistics.fmean(v) for v in groups.values()]
            elif dose == "all":
                values = [r[i] for i in finite]
                pooled_mean = statistics.fmean(values) if values else float("nan")
            else:
                values = [r[i] for i in finite if key_of(per_pair[i])[2] == float(dose)]
            mean = statistics.fmean(values) if values else float("nan")
            median = statistics.median(values) if values else float("nan")
            sb = 2 * mean / (1 + mean) if values else float("nan")
            if not (
                int(row["n_pairs"]) == len(values)
                and close(num(row["splithalf_mean_r"]), mean, 4)
                and close(num(row["splithalf_median_r"]), median, 4)
                and close(num(row["spearman_brown_full"]), sb, 4)
            ):
                bad.append(f"{dose}/{weighting}")
        n_rows = sum(1 for s in dose_strata if s["gene_set"] == label)
        verdict(
            f"{label}: every dose-strata row recomputes from the per-condition table",
            f"{n_rows} rows (per dose level, all triples, per line-drug)",
            f"{n_rows - len(bad)} of {n_rows} recompute" + (f"; disagreeing {bad}" if bad else ""),
            not bad and n_rows > 0,
        )
        verdict(
            f"{label}: the summary row's mean is the all-triples, equal-weight row",
            f"reported splithalf_mean_r {summary[f'{label}_splithalf_mean_r']}",
            f"all triples, equal weight: {pooled_mean:.4f}",
            close(num(summary[f"{label}_splithalf_mean_r"]), pooled_mean, 3),
        )

## Claim 14 — the responder-overlap table obeys its own set identities

The overlap diagnostic counts, per triple, the genes each half called and the genes both called.
`n_both` cannot exceed either count and the Jaccard overlap is `n_both / (n_first + n_second -
n_both)`; both are recomputed on every row, and there is one row per scored triple.

In [ ]:
overlap = read_rows("rung0_responder_overlap.csv")
if overlap and per_pair:
    bad = 0
    for row in overlap:
        first, second, both = num(row["n_first"]), num(row["n_second"]), num(row["n_both"])
        union = first + second - both
        jaccard = both / union if union > 0 else float("nan")
        if both > min(first, second) or not close(num(row["jaccard"]), jaccard, 4):
            bad += 1
    verdict(
        "the responder-overlap table obeys its set identities, one row per condition",
        "n_both <= min(n_first, n_second); jaccard = n_both / (n_first + n_second - n_both)",
        f"{len(overlap)} rows, {bad} break an identity; {len(per_pair)} scored triples",
        bad == 0 and len(overlap) == len(per_pair) > 0,
    )

## Claim 15 — every recorded checksum recomputes from the file it names

The most important cell in this notebook. The audit reads these artifacts in the working tree,
before they are committed (PROCESS, "What reaches GitHub, and when"), so nothing else ties what a
reader audited to what a reviewer later pulls. `audit_checksums.json` records the sha256 of every
table and figure the run wrote; recomputing them now is what closes that window. A single altered
byte anywhere moves a hash and fails here.

In [ ]:
checksums_path = TASK_DIR / "audit_checksums.json"
if checksums_path.exists():
    recorded = json.loads(checksums_path.read_text())
    by_name = {path.name: path for path in TASK_DIR.rglob("*") if path.is_file()}
    missing = sorted(name for name in recorded if name not in by_name)
    moved = sorted(
        name
        for name, digest in recorded.items()
        if name in by_name and sha256_of(by_name[name]) != digest
    )
    detail = f"{len(recorded) - len(missing) - len(moved)} of {len(recorded)} match"
    verdict(
        "every recorded artifact checksum recomputes from the file it names",
        f"{len(recorded)} sha256 entries in audit_checksums.json",
        (
            detail
            + (f"; missing {missing}" if missing else "")
            + (f"; CHANGED {moved}" if moved else "")
        ),
        not missing and not moved and bool(recorded),
    )
else:
    print(f"artifact not present yet: {checksums_path}")

## Claim 16 — the data pin: the tranche's content hash, recomputed from the committed manifest

The 1,026 shards are on cluster scratch and cannot be rehashed on a laptop. What can be rehashed is
the manifest committed beside the tranche record: `scripts/register_tranche.py` builds one
`relative path <tab> size <tab> sha256` line per shard and takes the sha256 of that text. Rebuilding
the text from the manifest and hashing it confirms the record and the manifest describe the same
1,026 files — the pin on which bytes the run read, without holding those bytes.

In [ ]:
record_path = REPO / "data" / "tranches" / f"{TRANCHE}.json"
manifest_path = REPO / "data" / "tranches" / f"{TRANCHE}.manifest.txt"
if record_path.exists() and manifest_path.exists():
    record = json.loads(record_path.read_text())
    lines = [line.split("\t") for line in manifest_path.read_text().splitlines()]
    text = "".join(f"{rel}\t{size}\t{sha}\n" for rel, size, sha in lines)
    recomputed = hashlib.sha256(text.encode()).hexdigest()
    verdict(
        "the tranche content hash recomputes from the committed manifest",
        f"record content_hash {record['content_hash']}",
        f"sha256 of the rebuilt manifest text {recomputed}",
        recomputed == record["content_hash"],
    )
    verdict(
        "the manifest describes the whole download",
        "1,026 shards (docs/DATA.md)",
        f"{len(lines)} manifest lines, each path/size/sha256",
        len(lines) == 1026 and all(len(line) == 3 for line in lines),
    )
else:
    print(f"artifact not present yet: {record_path}")

## Claim 17 — the permutation check, when it has run

Mismatched draws reuse the same half-profiles, so they are not independent and the bootstrap's
p-value could be optimistic. Permuting the pairing measures that dependence directly: each
permutation gives one null mean, and the exact p-value is `(1 + #{permutations at least as large as
the observed}) / (1 + #permutations)`. It is a separate cluster job — when its output is absent this
cell says so and moves on.

In [ ]:
for label, suffix in (("all", ""), ("responder", "_responder")):
    perm_summary = read_rows(f"rung0_permutation_summary{suffix}.csv")
    perm_means = read_rows(f"rung0_permutation_perm_means{suffix}.csv")
    if not (perm_summary and perm_means):
        continue
    row = perm_summary[0]
    means = [num(entry["perm_mean"]) for entry in perm_means]
    observed = num(row["observed_mean"])
    at_least = sum(1 for m in means if m >= observed)
    p_exact = (1 + at_least) / (1 + len(means))
    verdict(
        f"{label}: the permutation-mean summary recomputes from the draws",
        f"reported mean {row['perm_mean_mean']}, sd {row['perm_mean_sd']} over {row['n_perm']}",
        f"mean {statistics.fmean(means):.4f}, sd {statistics.stdev(means):.4f} over {len(means)}",
        len(means) == int(row["n_perm"])
        and close(num(row["perm_mean_mean"]), statistics.fmean(means), 4)
        and close(num(row["perm_mean_sd"]), statistics.stdev(means), 4),
    )
    verdict(
        f"{label}: the exact permutation p-value recomputes",
        f"reported p_exact {row['p_exact']}",
        f"(1 + {at_least}) / (1 + {len(means)}) = {p_exact:.4f}",
        close(num(row["p_exact"]), p_exact, 4),
    )
    verdict(
        f"{label}: the observed mean sits above every permutation",
        f"observed {observed} above the largest of {len(means)} permutation means",
        f"largest permutation mean {max(means):.4f}",
        observed > max(means),
    )

## Claim 18 — the promoted copies, once they exist

Promotion follows gate 2, so before it there is nothing under `results/` to disagree with and this
cell says so. Afterwards, each promoted table must be byte-identical to the task-side copy the
audit read, the provenance record's `result_sha256` must recompute from the promoted file, and the
record's `code_commit` must be the commit the RUN was made at — the `git_sha` in the run's own
parameter sidecar — with the promotion commit recorded separately. A record naming the promotion
commit instead names code that cannot reproduce the result.

In [ ]:
promoted_dir = REPO / "results" / TASK
records = sorted(promoted_dir.glob("*.provenance.json")) if promoted_dir.is_dir() else []
if TASK_DIR.resolve() != (REPO / "docs" / "tasks" / TASK).resolve():
    # The promoted copies belong to the real task folder. Comparing them against another
    # directory's tables asks whether two different runs produced the same bytes, which is
    # not a question about promotion.
    print(f"promotion claims apply to the task folder only; this run is {TASK_DIR}")
    records = []
elif not records:
    print(f"not promoted yet (promotion follows gate 2): {promoted_dir}")
for provenance in records:
    record = json.loads(provenance.read_text())
    promoted = REPO / record["result"]
    task_side = TASK_DIR / promoted.name
    identical = task_side.exists() and task_side.read_bytes() == promoted.read_bytes()
    verdict(
        f"{promoted.name}: the promoted copy is the table the audit read",
        "byte-identical to the task-side copy, and its recorded sha256 recomputes",
        f"identical {identical}; sha256 matches {sha256_of(promoted) == record['result_sha256']}",
        identical and sha256_of(promoted) == record["result_sha256"],
    )
    sidecar = TASK_DIR / (promoted.stem + ".params.json")
    run_sha = json.loads(sidecar.read_text()).get("git_sha") if sidecar.exists() else None
    recorded = record.get("environment", {}).get("code_commit", "")
    verdict(
        f"{promoted.name}: the record names the commit the run was made at",
        f"code_commit {recorded[:12]}, promotion_commit {str(record.get('promotion_commit'))[:12]}",
        f"the run's sidecar says git_sha {str(run_sha)[:12]}",
        bool(run_sha) and recorded == run_sha and bool(record.get("promotion_commit")),
    )

## Cross-check — the same claims through `scripts/verify_rung0.py`

Everything above was recomputed here, in the open. This last cell runs the project's verification
battery over the same artifacts, so the two paths are compared rather than one standing in for the
other. It is the notebook's cross-check, never its body: a notebook that only called this script
would relocate the trust instead of discharging it. The script exits 2 when the run has not
happened yet, which is what the message below reports on a fresh checkout.

In [ ]:
completed = subprocess.run(
    [sys.executable, str(REPO / "scripts" / "verify_rung0.py"), "--task-dir", str(TASK_DIR)],
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout or completed.stderr)
print(f"exit status {completed.returncode}")